In [67]:
import math
import time
import pandas as pd
import numpy as np

In [68]:
def distancia_haversine(lats1, lons1, lats2, lons2):
    """Versión vectorizada para calcular múltiples distancias simultáneamente"""
    R = 6371.0  # Radio de la Tierra en km
    lats1, lons1, lats2, lons2 = map(np.radians, [lats1, lons1, lats2, lons2])
    dlat = lats2 - lats1
    dlon = lons2 - lons1

    a = np.sin(dlat/2)**2 + np.cos(lats1) * np.cos(lats2) * np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

In [69]:
class Vertice:
    """Clase optimizada para representar vértices en el grafo"""
    __slots__ = ('valor', 'latitud', 'longitud', 'vecinosSalida', 'vecinosEntrada',
                 'tiempoEntrada', 'tiempoSalida', 'estado', 'padre', 'distEstimada')

    def __init__(self, valor, latitud=None, longitud=None):
        self.valor = valor
        self.latitud = latitud
        self.longitud = longitud
        self.vecinosSalida = {}
        self.vecinosEntrada = {}
        self.reset_attributes()

    def reset_attributes(self):
        """Reinicia atributos utilizados por algoritmos de recorrido"""
        self.tiempoEntrada = None
        self.tiempoSalida = None
        self.estado = "no_visitado"
        self.padre = None
        self.distEstimada = math.inf

    def tieneVecinoSalida(self, v):
        return v in self.vecinosSalida

    def tieneVecinoEntrada(self, v):
        return v in self.vecinosEntrada

    def tieneVecino(self, v):
        return self.tieneVecinoEntrada(v) or self.tieneVecinoSalida(v)

    def obtenerVecinosSalida(self):
        return list(self.vecinosSalida.keys())

    def obtenerVecinosEntrada(self):
        return list(self.vecinosEntrada.keys())

    def obtenerVecinosSalidaConPesos(self):
        return list(self.vecinosSalida.items())

    def obtenerVecinosEntradaConPesos(self):
        return list(self.vecinosEntrada.items())

    def agregarVecinoSalida(self, v, peso):
        self.vecinosSalida[v] = peso

    def agregarVecinoEntrada(self, v, peso):
        self.vecinosEntrada[v] = peso

    def __str__(self):
        return str(self.valor)

    def __repr__(self):
        return f"Vertice({self.valor}, lat={self.latitud}, lon={self.longitud})"


class Grafo:
    """Clase optimizada para representar grafos con vértices geolocalizados"""
    def __init__(self):
        self.vertices = []

    def agregarVertice(self, vertice):
        self.vertices.append(vertice)

    def agregarAristaDirigida(self, u, v, peso=1):
        u.agregarVecinoSalida(v, peso)
        v.agregarVecinoEntrada(u, peso)

    def agregarAristaBidireccional(self, u, v, peso=1):
        self.agregarAristaDirigida(u, v, peso)
        self.agregarAristaDirigida(v, u, peso)

    def obtenerAristasDirigidas(self):
        aristas = []
        for v in self.vertices:
            aristas.extend((v, destino, peso) for destino, peso in v.obtenerVecinosSalidaConPesos())
        return aristas

    def __str__(self):
        resultado = "Grafo con:\n"
        resultado += f"\tVertices: {len(self.vertices)}\n"
        resultado += f"\tAristas: {sum(len(v.vecinosSalida) for v in self.vertices)}\n"
        return resultado


In [70]:
def cargarUbicaciones(ruta_ubicacion, chunk_size=100000):
    """Carga optimizada de ubicaciones desde archivo CSV"""
    grafos = Grafo()
    dtype = {'latitud': np.float32, 'longitud': np.float32}

    for chunk in pd.read_csv(ruta_ubicacion, header=None,
                             names=['latitud', 'longitud'],
                             chunksize=chunk_size, dtype=dtype):
        indices = chunk.index + 1
        latitudes = chunk['latitud'].values
        longitudes = chunk['longitud'].values

        for idx, lat, lon in zip(indices, latitudes, longitudes):
            grafos.agregarVertice(Vertice(valor=idx, latitud=lat, longitud=lon))

    return grafos


In [71]:
def cargarUsuarios(ruta_usuarios, grafos):
    #Carga optimizada de relaciones entre usuarios
    start = time.time()
    vertices = grafos.vertices

    data = []
    with open(ruta_usuarios, 'r') as f:
        for line in f:
            if line.strip():
                data.append(np.fromstring(line, sep=',', dtype=np.int32))

    print(f"Lectura de Usuarios completada en: {time.time() - start:.4f}s")

    lats_u = np.array([v.latitud for v in vertices[:len(data)]], dtype=np.float32)
    lons_u = np.array([v.longitud for v in vertices[:len(data)]], dtype=np.float32)

    ini = time.time()
    total = len(data)

    for i, (u, vecinos) in enumerate(zip(vertices[:len(data)], data)):
        if vecinos.size == 0:
            continue

        vecinos_v = [vertices[v_idx - 1] for v_idx in vecinos]

        lats_v = np.array([v.latitud for v in vecinos_v], dtype=np.float32)
        lons_v = np.array([v.longitud for v in vecinos_v], dtype=np.float32)

        distancias = distancia_haversine(np.full_like(lats_v, lats_u[i]), np.full_like(lons_v, lons_u[i]), lats_v,lons_v)

        for v, dist in zip(vecinos_v, distancias):
            u.agregarVecinoSalida(v, dist)
            v.agregarVecinoEntrada(u, dist)

        if (i + 1) % 100000 == 0 or (i + 1) == total:
            porcentaje = (i + 1) * 100 / total
            print(f"Procesados {i+1}/{total} ({porcentaje:.2f}%) | Tiempo: {time.time()-ini:.2f}s", end='\r')

    print(f"\nProcesamiento completado en: {time.time() - ini:.2f}s")
    return grafos

In [72]:
print("Cargando ubicaciones...")
inicio = time.time()
grafos = cargarUbicaciones('datosLO.txt')
print(f"Tiempo total ubicaciones: {time.time() - inicio:.4f} segundos")
print(f"Vertices cargados: {len(grafos.vertices)}")

print("\nCargando usuarios y conexiones...")
inicio = time.time()
grafos = cargarUsuarios('datosUS.txt', grafos)
print(f"Tiempo total usuarios: {time.time() - inicio:.4f} segundos")

uno = time.time()
total_aristas = sum(len(v.vecinosSalida) for v in grafos.vertices)
print(f"Tiempo total vertices: {time.time()-uno:.4f}")

Cargando ubicaciones...
Tiempo total ubicaciones: 142.9400 segundos
Vertices cargados: 100000

Cargando usuarios y conexiones...
Lectura de Usuarios completada en: 6.8071s
Procesados 100000/100000 (100.00%) | Tiempo: 132.83s
Procesamiento completado en: 132.89s
Tiempo total usuarios: 142.1461 segundos
Tiempo total vertices: 0.0609


In [8]:
print(f"\nResumen del grafo:")
print(f"- Vertices: {len(grafos.vertices)}")
print(f"- Aristas: {total_aristas}")
print(f"- Densidad: {total_aristas / (len(grafos.vertices) * (len(grafos.vertices) - 1)):.6f}")


Resumen del grafo:
- Vertices: 100000
- Aristas: 49986140
- Densidad: 0.004999


In [ ]:
import plotly.graph_objects as go
import numpy as np

def visualizar_muestra_geografica(grafo, porcentaje=0.05, titulo="Visualización Aleatoria de Nodos", mostrar_hover=False):
    """
    Visualiza una muestra aleatoria de los vértices del grafo en un mapa geográfico (optimizado para alto volumen).
    
    Parámetros:
        grafo (Grafo): Grafo con vértices geolocalizados.
        porcentaje (float): Porcentaje de nodos a mostrar (ej. 0.05 para 5%).
        titulo (str): Título del gráfico.
        mostrar_hover (bool): Mostrar texto al pasar el mouse (desactivado por defecto).
    """
    vertices = grafo.vertices
    total_vertices = len(vertices)
    muestra_tamaño = max(1, int(total_vertices * porcentaje))

    print(f"Mostrando {muestra_tamaño:,} de {total_vertices:,} vértices ({porcentaje * 100:.2f}%)")

    # Optimización: convertir una vez a np array para acceso directo
    vertices_array = np.array(vertices, dtype=object)
    indices_muestra = np.random.choice(total_vertices, muestra_tamaño, replace=False)

    # Acceso vectorizado
    muestra = vertices_array[indices_muestra]
    latitudes = np.fromiter((v.latitud for v in muestra), dtype=np.float32)
    longitudes = np.fromiter((v.longitud for v in muestra), dtype=np.float32)

    # Solo si se desea texto flotante (no recomendable con millones)
    if mostrar_hover:
        nombres = [str(v.valor) for v in muestra]
        hover = 'text'
    else:
        nombres = None
        hover = 'skip'

    node_trace = go.Scattergeo(
        lat=latitudes,
        lon=longitudes,
        mode='markers',
        marker=dict(size=2, color='blue', opacity=0.5),
        text=nombres,
        hoverinfo=hover
    )

    fig = go.Figure(data=[node_trace])
    fig.update_layout(
        title=titulo,
        showlegend=False,
        geo=dict(
            projection_type="natural earth",
            showland=True,
            landcolor="rgb(240, 240, 240)",
            showcountries=False,
            lataxis=dict(range=[float(latitudes.min()), float(latitudes.max())]),
            lonaxis=dict(range=[float(longitudes.min()), float(longitudes.max())])
        ),
        margin=dict(l=0, r=0, t=30, b=0),
    )

    fig.show()

visualizar_muestra_geografica(grafos, porcentaje=0.02)  # 2%



In [22]:
visualizar_muestra_geografica(grafos, porcentaje=0.01)  # 2%


Mostrando 1,000 de 100,000 vértices (1.00%)


## Dijkstra

In [73]:
import heapq

def dijkstra(grafo: Grafo, id_origen: int):
    """
    Algoritmo de Dijkstra optimizado para grafos con vértices geolocalizados.
    Parámetro:
        grafo (Grafo): El grafo cargado.
        id_origen (int): ID del nodo origen (1-indexado).
    """

    # Restablecer todos los vértices
    for v in grafo.vertices:
        v.reset_attributes()

    origen = grafo.vertices[id_origen - 1]
    origen.distEstimada = 0

    # Cola de prioridad: (distancia acumulada, vertice)
    heap = [(0, origen)]

    while heap:
        dist_actual, actual = heapq.heappop(heap)

        if actual.estado == "visitado":
            continue
        actual.estado = "visitado"

        for vecino, peso in actual.obtenerVecinosSalidaConPesos():
            nueva_dist = dist_actual + peso
            if nueva_dist < vecino.distEstimada:
                vecino.distEstimada = nueva_dist
                vecino.padre = actual
                heapq.heappush(heap, (nueva_dist, vecino))

    print(f"\nDijkstra desde nodo {id_origen} completado.")
    return {v.valor: v.distEstimada for v in grafo.vertices}


In [74]:
def graficar_resultado_dijkstra(grafo: Grafo, id_origen: int):
    """
    Muestra en un mapa los resultados del algoritmo de Dijkstra.
    """
    origen = grafo.vertices[id_origen - 1]

    latitudes = []
    longitudes = []
    distancias = []

    for v in grafo.vertices:
        if v.distEstimada < math.inf:
            latitudes.append(v.latitud)
            longitudes.append(v.longitud)
            distancias.append(v.distEstimada)

    fig = go.Figure()

    # Nodos alcanzables (gradiente de distancia)
    fig.add_trace(go.Scattergeo(
        lon=longitudes,
        lat=latitudes,
        text=[f"ID: {v.valor}<br>Distancia: {v.distEstimada:.2f} km"
              for v in grafo.vertices if v.distEstimada < math.inf],
        mode='markers',
        marker=dict(
            size=4,
            color=distancias,
            colorscale='Viridis',
            colorbar=dict(title='Distancia (km)'),
            showscale=True
        ),
        name="Nodos alcanzables"
    ))

    # Nodo origen
    fig.add_trace(go.Scattergeo(
        lon=[origen.longitud],
        lat=[origen.latitud],
        text=f"Origen: {origen.valor}",
        mode='markers+text',
        marker=dict(size=10, color='red', symbol='star'),
        name="Nodo Origen"
    ))

    fig.update_layout(
        title=f'Distancias desde nodo origen {id_origen}',
        geo=dict(showland=True, landcolor="rgb(243, 243, 243)", showcountries=True),
        margin=dict(l=0, r=0, t=40, b=0)
    )

    fig.show()


In [ ]:
# Ejecutar Dijkstra desde un nodo específico (por ejemplo, nodo 1)
origen_usuario = int(input("\nIngrese el ID del nodo origen (por ejemplo 1): "))
resultado = dijkstra(grafos, origen_usuario)

# Mostrar distancias mínimas (opcional)
print("\nDistancias mínimas desde el nodo origen:")
for nodo_id in sorted(resultado)[:10]:
    dist = resultado[nodo_id]
    print(f"Distancia a nodo {nodo_id}: {dist:.4f} km" if dist < math.inf else f"Nodo {nodo_id} no alcanzable")

# Mostrar gráfico interactivo
graficar_resultado_dijkstra(grafos, origen_usuario)


## Comunidad

In [40]:
from collections import defaultdict
import random

def modularidad(comunidades, m, grados, adyacencia):
    """Calcula la modularidad Q del grafo dado un particionado en comunidades."""
    Q = 0.0
    for comunidad in comunidades.values():
        suma_interna = 0.0
        suma_total = 0.0
        for u in comunidad:
            suma_total += grados[u]
            for v, peso in adyacencia[u].items():
                if v in comunidad:
                    suma_interna += peso
        Q += (suma_interna / (2 * m)) - (suma_total / (2 * m)) ** 2
    return Q


In [61]:
from collections import defaultdict
import random
import time

def louvain_optimizado(grafo, delta_q_min=1e-5, cambio_minimo_porcentaje=0.01):
    vertices = grafo.vertices
    n = len(vertices)
    comunidad = {v: i for i, v in enumerate(vertices)}
    adyacencia = {v: v.vecinosSalida for v in vertices}
    grados = {v: sum(adyacencia[v].values()) for v in vertices}
    m2 = sum(grados.values())  # 2 * m (suma total de pesos de aristas)

    iteracion = 0
    while True:
        iteracion += 1
        cambios = 0
        orden = vertices.copy()
        random.shuffle(orden)

        for v in orden:
            c_actual = comunidad[v]
            mejor_comunidad = c_actual
            grado_v = grados[v]
            suma_adyacente = defaultdict(float)
            suma_comunidad = defaultdict(float)

            for u, peso in adyacencia[v].items():
                c = comunidad[u]
                suma_adyacente[c] += peso
                suma_comunidad[c] += grados[u]

            mejor_delta_q = 0.0
            for c in suma_adyacente:
                if c == c_actual:
                    continue
                ki_in = suma_adyacente[c]
                sum_tot = suma_comunidad[c]
                delta_q = ki_in - (grado_v * sum_tot) / m2
                if delta_q > mejor_delta_q and delta_q > delta_q_min:
                    mejor_delta_q = delta_q
                    mejor_comunidad = c

            if mejor_comunidad != c_actual:
                comunidad[v] = mejor_comunidad
                cambios += 1

        porcentaje_cambios = cambios / n
        print(f"Iteración {iteracion} — Cambios: {cambios} ({porcentaje_cambios:.4%})")
        if porcentaje_cambios < cambio_minimo_porcentaje:
            break

    resultado = defaultdict(list)
    for v, c in comunidad.items():
        resultado[c].append(v.valor)

    return resultado




In [62]:
print("Ejecutando Louvain optimizado...")
inicio = time.time()

comunidades = louvain_optimizado(grafos)

duracion = time.time() - inicio
print(f"\n✅ Detección completada en {duracion:.2f} segundos")
print(f"Comunidades encontradas: {len(comunidades)}")

# Mostrar primeras 3 comunidades
for i, (com, nodos) in enumerate(comunidades.items()):
    print(f"Comunidad {com} — {len(nodos)} nodos — Ej: {nodos[:5]}")
    if i == 2:
        break



Ejecutando Louvain optimizado...
Iteración 1 — Cambios: 100000 (100.0000%)
Iteración 2 — Cambios: 99940 (99.9400%)
Iteración 3 — Cambios: 99868 (99.8680%)
Iteración 4 — Cambios: 99834 (99.8340%)
Iteración 5 — Cambios: 99840 (99.8400%)
Iteración 6 — Cambios: 99839 (99.8390%)
Iteración 7 — Cambios: 99845 (99.8450%)
Iteración 8 — Cambios: 99836 (99.8360%)
Iteración 9 — Cambios: 99834 (99.8340%)
Iteración 10 — Cambios: 99840 (99.8400%)
Iteración 11 — Cambios: 99828 (99.8280%)
Iteración 12 — Cambios: 99869 (99.8690%)


KeyboardInterrupt: 